<a href="https://colab.research.google.com/github/Dhlih/Submission-Dicoding-Analisis-Sentimen/blob/main/Inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Import Library

In [42]:
import pandas as pd  # Pandas untuk manipulasi dan analisis data
pd.options.mode.chained_assignment = None  # Menonaktifkan peringatan chaining

import numpy as np  # NumPy untuk komputasi numerik
import matplotlib.pyplot as plt  # Matplotlib untuk visualisasi data
import seaborn as sns  # Seaborn untuk visualisasi data statistik, mengatur gaya visualisasi
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix  # Evaluasi model klasifikasi

import re  # Modul untuk bekerja dengan ekspresi reguler
import string  # Berisi konstanta string, seperti tanda baca

from nltk.tokenize import word_tokenize  # Tokenisasi teks
from nltk.corpus import stopwords  # Daftar kata-kata berhenti dalam teks

!pip install emoji

import nltk  # Import pustaka NLTK (Natural Language Toolkit).
nltk.download('punkt_tab')  # Mengunduh dataset yang diperlukan untuk tokenisasi teks.

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

## Preprocess Text

In [43]:
import emoji

def cleaningText(text):
    # 1. Ubah emoji menjadi teks (Misal: 😂 menjadi :face_with_tears_of_joy:)
    text = emoji.demojize(text, delimiters=(" ", " "))

    # 2. Pembersihan standar
    text = re.sub(r'@[A-Za-z0-9]+', '', text) # menghapus mention
    text = re.sub(r'#[A-Za-z0-9]+', '', text) # menghapus hashtag
    text = re.sub(r'RT[\s]', '', text)         # menghapus RT
    text = re.sub(r"http\S+", '', text)       # menghapus link
    text = re.sub(r'[0-9]+', '', text)         # menghapus angka

    # 3. Menghapus karakter selain huruf, angka, spasi, dan UNDERSCORE
    text = re.sub(r'[^\w\s]', '', text)

    text = text.replace('\n', ' ') # mengganti baris baru dengan spasi

    # 4. Menghapus tanda baca (kecuali underscore agar label emoji tetap utuh)
    punc_clean = string.punctuation.replace('_', '')
    text = text.translate(str.maketrans('', '', punc_clean))

    # 5. Merapikan spasi berlebih
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()

    return text

def casefoldingText(text): # Mengubah semua karakter dalam teks menjadi huruf kecil
    text = text.lower()
    return text

def tokenizingText(text): # Memecah atau membagi string, teks menjadi daftar token
    text = word_tokenize(text)
    return text

def toSentence(list_words): # Mengubah daftar kata menjadi kalimat
    sentence = ' '.join(word for word in list_words)
    return sentence

In [44]:
import json

with open('slang_words.json', 'r') as f:
    slang_words = json.load(f)

def fix_slangwords(text):
    words = text.split()
    fixed_words = []

    for word in words:
        if word.lower() in slang_words:
            fixed_words.append(slang_words[word.lower()])
        else:
            fixed_words.append(word)

    fixed_text = ' '.join(fixed_words)
    return fixed_text

with open('stop_words.txt', 'r') as f:
    stop_words = set(f.read().splitlines())

def filteringText(text): # Menghapus stopwords dalam teks
    filtered = []
    for txt in text:
        if txt not in stop_words:
            filtered.append(txt)
    text = filtered
    return text

In [45]:
def preprocess_text(text):
    # 1. Cleaning & Demojize (Output: String)
    text = cleaningText(text)

    # 2. Case Folding (Output: String)
    text = casefoldingText(text)

    # 3. Fix Slangwords (Output: String)
    text = fix_slangwords(text)

    # 4. Tokenizing (Output: List of words)
    tokens = tokenizingText(text)

    # 5. Filtering/Stopwords (Output: List of words)
    filtered_tokens = filteringText(tokens)

    # 6. toSentence (Output: String)
    final_text = toSentence(filtered_tokens)

    return final_text

## Inference

In [53]:
import joblib

# 1. Load model dan vectorizer yang sudah Anda simpan sebelumnya
model_terlatih = joblib.load('model_sentimen_ipusnas.pkl')
tfidf_vectorizer = joblib.load('tfidf_vectorizer.pkl')

def predict_sentimen(input_teks):
    """
    Fungsi untuk menerima input teks ulasan baru dan mengeluarkan label sentimen.
    """
    # Langkah A: Preprocessing
    teks_bersih = preprocess_text(input_teks)

    # Langkah B: Feature Extraction
    vektor_teks = tfidf_vectorizer.transform([teks_bersih])

    # Langkah C: Prediksi menggunakan Model (Linear SVC / Logistic Regression)
    prediksi_label = model_terlatih.predict(vektor_teks)[0]

    if prediksi_label == 0:
      return "negatif"
    elif prediksi_label == 1:
      return "positif"

# --- Contoh Penggunaan ---
ulasan_1 = "Antrean pinjam bukunya lama banget, tolong diperbaiki sistemnya!"
hasil_1 = predict_sentimen(ulasan_1)

ulasan_2 = "Aplikasi ini sangat bermanfaat bagi saya, terima kasih"
hasil_2 = predict_sentimen(ulasan_2)

ulasan_3 = "Aplikasi gak jelas, sering banget error entah kenapa"
hasil_3 = predict_sentimen(ulasan_2)

print(f"Ulasan 1: {ulasan_1}")
print(f"Hasil Prediksi Sentimen: {hasil_1}\n")

print(f"Ulasan 2: {ulasan_2}")
print(f"Hasil Prediksi Sentimen: {hasil_2}\n")

print(f"Ulasan 3: {ulasan_3}")
print(f"Hasil Prediksi Sentimen: {hasil_3}")

Ulasan 1: Antrean pinjam bukunya lama banget, tolong diperbaiki sistemnya!
Hasil Prediksi Sentimen: negatif

Ulasan 2: Aplikasi ini sangat bermanfaat bagi saya, terima kasih
Hasil Prediksi Sentimen: positif

Ulasan 3: Aplikasi gak jelas, sering banget error entah kenapa
Hasil Prediksi Sentimen: positif
